# Analyse des correspondances multiples (ACM)

Cette partie construit une **ACM globale** sur un ensemble cohérent de variables qualitatives décrivant les formations Parcoursup.

L'objectif est d'aller plus loin qu'une lecture en simple plan 1-2 :

- on prépare des variables qualitatives d'analyse à partir des indicateurs métiers ;
- on distingue clairement **variables actives** et **variables supplémentaires** ;
- on étudie plusieurs sous-espaces factoriels : `(1,2)`, `(1,3)`, `(2,3)` et `(3,4)` ;
- on interprète les axes à partir des **contributions**, du **cos2** et des variables les plus structurantes.

Le fil directeur reste celui du cours : une dimension ne s'interprète pas uniquement avec son inertie, mais surtout avec les modalités **bien représentées** et **fortement contributives**.


In [ ]:
source("../R/shared_data.R")
suppressPackageStartupMessages({
  library(tidyverse)
  library(FactoMineR)
  library(factoextra)
})

theme_set(theme_minimal(base_size = 12))

data_features_core <- load_analysis_data()


## 1. Construction d'un jeu de variables adapté à l'ACM

Les données sources contiennent peu de variables qualitatives directement exploitables pour une ACM riche. Pour construire une analyse plus pertinente, on transforme plusieurs indicateurs numériques en classes :

- pression de candidature ;
- taux d'admission ;
- taux d'accès ;
- niveau académique des admis via le score de mentions ;
- poids des boursiers parmi les admis ;
- taille des formations ;
- féminisation des admis ;
- profil dominant des néo-bacheliers admis.

Cette étape permet de mener une ACM sur une **description globale des formations**, et non sur de simples couples de variables.


In [ ]:
add_na_level <- function(x, level = "Non renseigne") {
  x <- as.factor(x)
  if (!(level %in% levels(x))) {
    levels(x) <- c(levels(x), level)
  }
  x[is.na(x)] <- level
  x
}

quartile_factor <- function(x, labels) {
  factor(
    dplyr::ntile(x, length(labels)),
    levels = seq_along(labels),
    labels = labels,
    ordered = TRUE
  )
}

mode_majoritaire <- function(gen, tech, pro, seuil_mixte = 0.45) {
  maxi <- pmax(gen, tech, pro, na.rm = TRUE)

  res <- case_when(
    is.na(maxi) ~ NA_character_,
    maxi < seuil_mixte ~ "Profil mixte",
    gen >= tech & gen >= pro ~ "Dominante generale",
    tech >= gen & tech >= pro ~ "Dominante technologique",
    TRUE ~ "Dominante professionnelle"
  )

  factor(
    res,
    levels = c(
      "Dominante generale",
      "Dominante technologique",
      "Dominante professionnelle",
      "Profil mixte"
    )
  )
}

acm_data <- data_features_core %>%
  transmute(
    selectivite = Sélectivité,
    statut = Statut.Etablissement,
    filiere = add_na_level(Filière.de.formation.très.agrégée),
    pression_niveau = quartile_factor(
      pression_candidature,
      c("Pression faible", "Pression moyenne", "Pression forte", "Pression tres forte")
    ),
    admission_niveau = quartile_factor(
      taux_admission,
      c("Admission tres faible", "Admission faible", "Admission elevee", "Admission tres elevee")
    ),
    acces_niveau = quartile_factor(
      taux_acces_num,
      c("Acces tres difficile", "Acces difficile", "Acces accessible", "Acces tres accessible")
    ),
    mention_niveau = quartile_factor(
      score_mention,
      c("Mentions faibles", "Mentions intermediaires", "Mentions elevees", "Mentions tres elevees")
    ),
    boursiers_niveau = quartile_factor(
      Pourcentage.boursiers.admis,
      c(
        "Boursiers peu presents",
        "Boursiers assez presents",
        "Boursiers presents",
        "Boursiers tres presents"
      )
    ),
    feminisation = case_when(
      pct_filles_admises < 0.40 ~ "Majoritairement masculine",
      pct_filles_admises <= 0.60 ~ "Mixte",
      TRUE ~ "Majoritairement feminine"
    ) %>%
      factor(
        levels = c(
          "Majoritairement masculine",
          "Mixte",
          "Majoritairement feminine"
        )
      ),
    profil_bac = mode_majoritaire(
      X..d.admis.néo.bacheliers.généraux / 100,
      X..d.admis.néo.bacheliers.technologiques / 100,
      X..d.admis.néo.bacheliers.professionnels / 100
    ),
    taille_formation = quartile_factor(
      Capacité.de.l.établissement.par.formation,
      c("Petite capacite", "Capacite moyenne", "Grande capacite", "Tres grande capacite")
    ),
    region = add_na_level(Région.de.l.établissement),
    remplissage_niveau = quartile_factor(
      taux_remplissage,
      c(
        "Remplissage faible",
        "Remplissage moyen",
        "Remplissage eleve",
        "Remplissage tres eleve"
      )
    ),
    boursiers_candidats_niveau = quartile_factor(
      Pourcentage.boursiers.candidats,
      c(
        "Peu de candidats boursiers",
        "Part moderee de boursiers",
        "Part elevee de boursiers",
        "Tres forte part de boursiers"
      )
    ),
    pression_candidature,
    taux_admission,
    taux_acces_num,
    taux_remplissage,
    score_mention,
    Pourcentage.boursiers.admis,
    pct_filles_admises,
    log_candidats,
    log_admis,
    log_capacite
  )

glimpse(acm_data)

active_vars <- c(
  "selectivite",
  "statut",
  "filiere",
  "pression_niveau",
  "admission_niveau",
  "acces_niveau",
  "mention_niveau",
  "boursiers_niveau",
  "feminisation",
  "profil_bac",
  "taille_formation"
)

quali_sup_vars <- c(
  "region",
  "remplissage_niveau",
  "boursiers_candidats_niveau"
)

quanti_sup_vars <- c(
  "pression_candidature",
  "taux_admission",
  "taux_acces_num",
  "taux_remplissage",
  "score_mention",
  "Pourcentage.boursiers.admis",
  "pct_filles_admises",
  "log_candidats",
  "log_admis",
  "log_capacite"
)


## 2. Variables actives et variables supplémentaires

On retient ici une logique analytique claire :

- les **variables actives** définissent l'espace factoriel ;
- les **variables qualitatives supplémentaires** servent à enrichir la lecture sans orienter la construction des axes ;
- les **variables quantitatives supplémentaires** permettent de raccrocher les dimensions à des grandeurs métiers interprétables.

Ce choix est important pour éviter qu'une variable de contexte ne domine artificiellement l'ACM.


In [ ]:
tibble(
  role = c(
    rep("Active", length(active_vars)),
    rep("Qualitative supplementaire", length(quali_sup_vars)),
    rep("Quantitative supplementaire", length(quanti_sup_vars))
  ),
  variable = c(active_vars, quali_sup_vars, quanti_sup_vars)
)

purrr::walk(
  active_vars,
  ~ {
    cat("\n###", .x, "\n")
    print(acm_data %>% count(.data[[.x]], sort = TRUE))
  }
)


## 3. Estimation de l'ACM globale

L'ACM est calculée sur l'ensemble des variables actives.  
Les variables supplémentaires sont projetées a posteriori pour enrichir l'interprétation.

Comme en cours, on ne se contente pas du premier plan factoriel :

- on regarde l'inertie expliquée par chaque axe ;
- on suit l'inertie cumulée quand on augmente le nombre de dimensions ;
- on conserve ensuite plusieurs plans de lecture.


In [ ]:
mca_input <- acm_data[, c(active_vars, quali_sup_vars, quanti_sup_vars)]

res_mca <- MCA(
  mca_input,
  quali.sup = (length(active_vars) + 1):(length(active_vars) + length(quali_sup_vars)),
  quanti.sup = (length(active_vars) + length(quali_sup_vars) + 1):ncol(mca_input),
  graph = FALSE
)

eig_tbl <- as_tibble(res_mca$eig[1:10, , drop = FALSE], rownames = "dimension")
colnames(eig_tbl) <- c("dimension", "valeur_propre", "pct_inertie", "pct_cumule")
eig_tbl

comparaison_dimensions <- eig_tbl %>%
  mutate(nb_dimensions = row_number()) %>%
  select(nb_dimensions, pct_inertie, pct_cumule)

comparaison_dimensions


### Lecture attendue

Dans une ACM, l'inertie par axe est souvent modeste, surtout quand il y a beaucoup de modalités.  
Il faut donc éviter deux pièges :

- rejeter un axe uniquement parce que son pourcentage d'inertie paraît faible ;
- interpréter un axe sans vérifier la contribution et la qualité de représentation des modalités.

La suite du notebook combine donc systématiquement :

- **inertie** ;
- **eta2 des variables actives** ;
- **contributions des modalités** ;
- **cos2**, pour vérifier que les modalités commentées sont bien représentées.


In [ ]:
fviz_screeplot(res_mca, addlabels = TRUE, ylim = c(0, max(eig_tbl$pct_inertie) * 1.15))

ggplot(comparaison_dimensions, aes(x = nb_dimensions, y = pct_cumule)) +
  geom_line(color = "#15616D", linewidth = 1) +
  geom_point(color = "#15616D", size = 2.5) +
  scale_x_continuous(breaks = comparaison_dimensions$nb_dimensions) +
  labs(
    title = "Inertie cumulee selon le nombre de dimensions conservees",
    x = "Nombre de dimensions",
    y = "Pourcentage cumule d'inertie"
  )


In [ ]:
top_modalites <- function(res, dim, n = 10, min_cos2 = 0.02) {
  tibble(
    modalite = rownames(res$var$coord),
    coord = res$var$coord[, dim],
    contrib = res$var$contrib[, dim],
    cos2 = res$var$cos2[, dim]
  ) %>%
    filter(cos2 >= min_cos2) %>%
    arrange(desc(contrib)) %>%
    slice_head(n = n)
}

top_variables <- function(res, dim, vars, n = 6) {
  tibble(
    variable = vars,
    eta2 = res$eta2[vars, dim]
  ) %>%
    arrange(desc(eta2)) %>%
    slice_head(n = n)
}

synthese_dimension <- function(res, dim, vars, n_var = 4, n_mod = 4) {
  vars_tbl <- top_variables(res, dim, vars, n = n_var)

  mod_tbl <- tibble(
    modalite = rownames(res$var$coord),
    coord = res$var$coord[, dim],
    contrib = res$var$contrib[, dim],
    cos2 = res$var$cos2[, dim]
  ) %>%
    filter(cos2 >= 0.02) %>%
    arrange(desc(contrib))

  pos_tbl <- mod_tbl %>%
    filter(coord > 0) %>%
    arrange(desc(coord), desc(contrib)) %>%
    slice_head(n = n_mod)

  neg_tbl <- mod_tbl %>%
    filter(coord < 0) %>%
    arrange(coord, desc(contrib)) %>%
    slice_head(n = n_mod)

  cat("\n==============================\n")
  cat("Dimension", dim, "\n")
  cat("==============================\n")
  cat("Variables les plus liees a l'axe :\n")
  print(vars_tbl)
  cat("\nModalites du pole negatif :\n")
  print(neg_tbl)
  cat("\nModalites du pole positif :\n")
  print(pos_tbl)
}

eta2_tbl <- as_tibble(res_mca$eta2[active_vars, 1:5, drop = FALSE], rownames = "variable")
eta2_tbl


## 4. Cas 1 : lecture classique du plan (1, 2)

On commence par le plan principal, mais sans s'y limiter.  
Le but ici est d'identifier les grandes oppositions structurelles entre formations :

- quelles variables expliquent le plus l'axe 1 ;
- quelles variables expliquent le plus l'axe 2 ;
- quelles modalités s'opposent sur ce plan.


In [ ]:
fviz_mca_var(
  res_mca,
  axes = c(1, 2),
  repel = TRUE,
  select.var = list(contrib = 25),
  col.var = "contrib"
)

top_variables(res_mca, 1, active_vars, n = 8)
top_modalites(res_mca, 1, n = 12)

top_variables(res_mca, 2, active_vars, n = 8)
top_modalites(res_mca, 2, n = 12)

synthese_dimension(res_mca, 1, active_vars)
synthese_dimension(res_mca, 2, active_vars)


### Interprétation méthodologique

Le plan `(1,2)` donne une première lecture globale, mais il ne résume pas toute la structure des données.

Dans cette ACM, il faudra surtout vérifier si :

- l'axe 1 capte une opposition entre formations **très demandées / peu accessibles** et formations **plus ouvertes** ;
- l'axe 2 traduit davantage la **composition sociale / scolaire** ou la **nature des formations** ;
- certaines modalités de filière restent visibles uniquement sur des axes plus profonds.

C'est précisément pour cela que l'on poursuit avec d'autres plans factoriels.


## 5. Cas 2 : aller au-delà du plan principal

Pour éviter une lecture trop courte, on explore plusieurs plans complémentaires :

- `(1,3)` pour voir ce que l'axe 3 apporte au premier grand clivage ;
- `(2,3)` pour croiser la deuxième et la troisième structuration ;
- `(3,4)` pour détecter des oppositions plus fines, parfois masquées sur le plan principal.

Cette étape répond directement au besoin d'une analyse **multi-dimensionnelle** et plus rigoureuse.


In [ ]:
plans_a_explorer <- list(
  c(1, 3),
  c(2, 3),
  c(3, 4)
)

purrr::walk(
  plans_a_explorer,
  ~ print(
    fviz_mca_var(
      res_mca,
      axes = .x,
      repel = TRUE,
      select.var = list(contrib = 20),
      col.var = "contrib"
    ) +
      labs(title = paste("Plan factoriel", paste(.x, collapse = "-")))
  )
)

for (dim in 3:5) {
  cat("\n\n########## ANALYSE DE LA DIMENSION", dim, "##########\n")
  print(top_variables(res_mca, dim, active_vars, n = 8))
  print(top_modalites(res_mca, dim, n = 12))
  synthese_dimension(res_mca, dim, active_vars)
}


### Ce qu'on cherche sur les dimensions 3, 4 et 5

Les dimensions supplémentaires peuvent faire apparaître des structures plus fines que le plan principal absorbe mal :

- différenciation entre types de filières ;
- opposition entre profils d'admis généraux, technologiques ou professionnels ;
- segmentation selon la féminisation, la taille de la formation ou le poids des boursiers.

Autrement dit, si une variable est peu visible sur `(1,2)` mais fortement liée à l'axe 3 ou 4, elle ne doit pas être écartée de l'interprétation globale.


## 6. Projection des variables supplémentaires

Les variables supplémentaires ne construisent pas les axes, mais elles aident à comprendre ce qu'ils recouvrent réellement.

On projette ici :

- des variables qualitatives de contexte : région, remplissage, poids des boursiers parmi les candidats ;
- des variables quantitatives : pression, admission, accès, score de mentions, taille et volumes.

Cette lecture croisée permet de raccrocher les dimensions à des phénomènes métiers concrets.


In [ ]:
quali_sup_eta2 <- as_tibble(
  res_mca$quali.sup$eta2[, 1:5, drop = FALSE],
  rownames = "variable"
)
quali_sup_eta2

quanti_sup_coord <- as_tibble(
  res_mca$quanti.sup$coord[, 1:5, drop = FALSE],
  rownames = "variable"
)
quanti_sup_coord

quanti_sup_coord %>%
  pivot_longer(-variable, names_to = "dimension", values_to = "coord") %>%
  ggplot(aes(x = dimension, y = variable, fill = coord)) +
  geom_tile() +
  scale_fill_gradient2(low = "#B23A48", mid = "white", high = "#2A9D8F") +
  labs(
    title = "Variables quantitatives supplementaires et axes de l'ACM",
    x = "Dimension",
    y = NULL,
    fill = "Coord."
  )


## 7. Individus : prudence et sélection des observations lisibles

Avec plus de 14 000 observations, la carte de tous les individus devient rapidement illisible.  
On privilégie donc :

- les individus les mieux représentés sur les premiers axes ;
- les plus contributeurs ;
- éventuellement un échantillon visuel pour éviter la surimpression.

Cette démarche est plus rigoureuse qu'un nuage massif difficilement interprétable.


In [ ]:
individus_bien_representes <- tibble(
  id = seq_len(nrow(res_mca$ind$coord)),
  coord_dim1 = res_mca$ind$coord[, 1],
  coord_dim2 = res_mca$ind$coord[, 2],
  cos2_12 = rowSums(res_mca$ind$cos2[, 1:2, drop = FALSE]),
  contrib_12 = rowSums(res_mca$ind$contrib[, 1:2, drop = FALSE])
) %>%
  arrange(desc(cos2_12))

individus_bien_representes %>% slice_head(n = 10)

set.seed(123)
ids_plot <- individus_bien_representes %>%
  slice_head(n = 800) %>%
  pull(id)

fviz_mca_ind(
  res_mca,
  axes = c(1, 2),
  select.ind = ids_plot,
  alpha.ind = 0.35,
  habillage = acm_data$selectivite[ids_plot],
  palette = c("#2A9D8F", "#E76F51"),
  addEllipses = TRUE
)


## 8. Synthèse finale

Cette ACM doit être lue comme une **cartographie multidimensionnelle des formations**.

Les points à retenir après exécution du notebook sont les suivants :

1. le plan `(1,2)` donne la structure la plus visible, mais il ne suffit pas à lui seul ;
2. les dimensions 3 à 5 servent à récupérer des oppositions plus fines, souvent liées au type de filière, au profil des admis ou à la composition sociale ;
3. les modalités à commenter en priorité sont celles qui cumulent :
   - contribution élevée ;
   - cos2 satisfaisant ;
   - position nette sur l'axe ;
4. les variables supplémentaires permettent de rattacher les axes à des indicateurs concrets du jeu de données.

On obtient ainsi une analyse plus cohérente qu'une suite d'ACM sur paires de variables, car l'espace factoriel résume ici **un système de caractéristiques simultanées**.
